# Leftover Lens: train on Village plates (Colab, T4 GPU)
Needs in the top level of My Drive: `leftover-lens.zip` and `village-dataset.zip` (written by `colab_autolabel.ipynb`).

1. Unpack the repo and the dataset.
2. Add copy-paste composites to the **training split only** (validation stays real, held-out plates).
3. Fine-tune YOLO26n-seg (from the Voxel51 food checkpoint if present).
4. Evaluate on the held-out plates: `eval.segmentation` writes the numbers to files. Those files, not anything typed by hand, go in the README.
5. Export ONNX for the rig.

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)
REPO_ZIP = "/content/drive/MyDrive/leftover-lens.zip"
DATA_ZIP = "/content/drive/MyDrive/village-dataset.zip"
DATA = "/content/datasets/village"
!rm -rf /content/leftover-lens && unzip -q -o {REPO_ZIP} -d /content
!rm -rf {DATA} && mkdir -p {DATA} && unzip -q -o {DATA_ZIP} -d {DATA}
%cd /content/leftover-lens
!pip -q install -e . ultralytics
!python -c "import ultralytics; print('ultralytics', ultralytics.__version__)"
!ls {DATA}/images/train | wc -l; ls {DATA}/images/val | wc -l

## 2. Composites (training split only)

In [ ]:
!python -m augment.dataset --data {DATA} --n 200 --seed 0
!ls {DATA}/images/train | wc -l

In [ ]:
# Look at a few composites: pasted food should sit on the plate and the masks should match it.
import glob
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

from augment.dataset import read_yolo_labels
from core.types import SEG_CLASSES

rng = np.random.RandomState(1)
colors = {c: rng.randint(60, 255, 3) for c in SEG_CLASSES}
files = sorted(glob.glob(f"{DATA}/images/train/comp_*.jpg"))[:8]
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax in axes.ravel():
    ax.axis("off")
for ax, f in zip(axes.ravel(), files, strict=False):
    img = cv2.imread(f)
    h, w = img.shape[:2]
    label_path = Path(f.replace("/images/", "/labels/")).with_suffix(".txt")
    labels = read_yolo_labels(label_path.read_text(), h, w)
    over = img.copy()
    for m in labels:
        over[m.mask] = (0.5 * over[m.mask] + 0.5 * colors[m.class_name]).astype(np.uint8)
    ax.imshow(over[:, :, ::-1])
    ax.set_title(", ".join(sorted({m.class_name for m in labels})), fontsize=8)
plt.tight_layout()
plt.show()

## 3. Train
Starts from the Voxel51 checkpoint (same 23-class list, already trained on food) when it is in Drive, otherwise from the COCO-pretrained `yolo26n-seg.pt`. Runs as a separate process so an out-of-memory kill does not take the notebook down. About 15-25 minutes on a T4.

In [ ]:
RUNS = "/content/drive/MyDrive/leftover-runs"
start = Path(f"{RUNS}/voxel51-v0/weights/best.pt")
MODEL = str(start) if start.exists() else "yolo26n-seg.pt"
print("starting from", MODEL)
ARGS = (
    f"model='{MODEL}', data='{DATA}/data.yaml', project='{RUNS}', name='village-v1', "
    "epochs=100, patience=25, batch=8, workers=2, exist_ok=True"
)
!python -c "from vision.train import train; print(train({ARGS}))"

## 4. Evaluate on the held-out plates
Writes `village-v1.json` and `village-v1.md` to `My Drive/leftover-runs/eval/`. Send those two files back; they go into `eval/results/` in the repo.

In [ ]:
BEST = f"{RUNS}/village-v1/weights/best.pt"
EVAL_ARGS = f"--weights {BEST} --data {DATA}/data.yaml --name village-v1 --out {RUNS}/eval"
!python -m eval.segmentation {EVAL_ARGS}

## 5. Export ONNX

In [ ]:
import shutil

from vision.export_onnx import export_onnx

onnx_path = export_onnx(Path(BEST))
shutil.copy(onnx_path, f"{RUNS}/village-v1.onnx")
print("saved MyDrive/leftover-runs/village-v1.onnx")

## 6. Look at predictions on held-out plates

In [ ]:
from IPython.display import Image, display
from ultralytics import YOLO

val = sorted(glob.glob(f"{DATA}/images/val/*.jpg"))
preds = YOLO(BEST).predict(
    val, conf=0.25, save=True, project="/content/preds", name="village-v1", exist_ok=True
)
for f in sorted(glob.glob("/content/preds/village-v1/*.jpg"))[:8]:
    display(Image(f, width=420))